# Quality sign-off

Task 10, Week 3, Phase 2. By this point the matching model (Task 7), the payment flow
(Task 9), and the revenue dashboard have all landed. This notebook is the final check
before calling the AI side of the platform done: does the same matching model, unchanged,
still recommend the right jobs now that everything else around it has shipped?

Order of work:
1. Load the model (unchanged from Task 7) and the integrated-platform sample (new, unseen data)
2. Run the quality report - one row per student, with explanation
3. Metrics vs the Task 7 baseline
4. Prove the sign-off process actually catches a real regression
5. One live demo walkthrough + the final sign-off report


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import joblib

from matching import TfidfMatcher, parse_skills, rank_jobs
from signoff import run_quality_report, precision_recall_fpr, load_task7_baseline, decide

pd.set_option('display.max_colwidth', 60)


## 1. The model and the data

`models/matching_model.pkl` is the exact file from Task 7 - not retrained, not re-tuned.
`data/students.csv` is 800 fresh student profiles representing live platform traffic -
none of these students were used to build or tune the model, and there are roughly 2.5x
as many of them as the entire Task 7 dataset, to actually test 'at scale' rather than just claim it.


In [ ]:
bundle = joblib.load('../models/matching_model.pkl')
matcher = TfidfMatcher.__new__(TfidfMatcher)
matcher.jobs_df = bundle['jobs_df']
matcher.job_skill_lists = bundle['job_skill_lists']
matcher.vectorizer = bundle['vectorizer']
matcher.job_matrix = bundle['job_matrix']
alpha = bundle['alpha']

students_df = pd.read_csv('../data/students.csv')
print(f'model alpha: {alpha}')
print(f'students in this sample: {len(students_df)} (ids {students_df.student_id.min()}-{students_df.student_id.max()})')
students_df.head()


## 2. Quality report - one row per student

Top recommendation, the score, whether it actually landed in the right role, and the
matched/missing skills behind that score. This is `reports/quality_report.csv`.


In [ ]:
quality_df = run_quality_report(matcher, alpha, students_df)
quality_df.head()


In [ ]:
print('top-1 recommendation accuracy:', round(quality_df['top1_correct_role'].mean() * 100, 1), '%')
quality_df['status'].value_counts()


## 3. Metrics vs the Task 7 baseline

Baseline here isn't invented for this report - it's pulled straight from Task 7's own
held-out test-split numbers (`data/task7_baseline_experiment_log.csv`), the last trusted
measurement of this model before any monetization work touched the platform.


In [ ]:
baseline_metrics = load_task7_baseline()
current_metrics = precision_recall_fpr(quality_df, matcher, students_df, alpha)

pd.DataFrame([
 {'stage': 'task7_baseline (held-out test split, n=46)', **baseline_metrics},
 {'stage': 'task10_integrated_platform_sample (n=800)', **current_metrics},
])


Precision and recall move down slightly - expected, given this sample is ~17x bigger
and intentionally messier (wider skill-coverage spread than Task 7's data_gen.py) - but
both stay within the tolerance the sign-off check uses, so this doesn't read as the model
actually getting worse, just as noisier real-world data being noisier.


## 4. Proving the sign-off check actually catches a regression

A check that always says APPROVED is worthless. So: corrupt 30% of the job postings'
required skills (simulating what a botched migration during the dashboard integration
could realistically do to the job feed) and confirm the same pipeline correctly flags it.


In [ ]:
import random

jobs_df = pd.read_csv('../data/jobs.csv')
broken_jobs = jobs_df.copy()
random.seed(1)
idx = broken_jobs.sample(frac=0.3, random_state=1).index
shuffled = broken_jobs.loc[idx, 'required_skills'].sample(frac=1, random_state=2).values
broken_jobs.loc[idx, 'required_skills'] = shuffled

broken_matcher = TfidfMatcher(broken_jobs)
broken_quality_df = run_quality_report(broken_matcher, alpha, students_df)
broken_metrics = precision_recall_fpr(broken_quality_df, broken_matcher, students_df, alpha)
broken_verdict = decide(baseline_metrics, broken_metrics, broken_quality_df)

print(broken_metrics)
print(broken_verdict)


`REJECTED`, as it should be - precision@5 drops well past tolerance once the job feed is
actually broken. Good, the sign-off isn't a rubber stamp. Back to the real (unbroken) data
for the rest of this notebook.


## 5. Live demo walkthrough


In [ ]:
demo_row = quality_df[quality_df['status'] == 'ok'].sample(1, random_state=11).iloc[0]
print(f"Student {demo_row['student_id']} (intended role: {demo_row['target_role']})")
print()
print(f"Top recommendation: {demo_row['top1_job']}  (score {demo_row['top1_score']})")
print(f"Matched: {demo_row['matched_skills']}")
print(f"Missing: {demo_row['missing_skills'] if demo_row['missing_skills'] else '(none)'}")
print(f"Correct role: {demo_row['top1_correct_role']}")


## Final sign-off report

In [ ]:
verdict = decide(baseline_metrics, current_metrics, quality_df)
symbol = '✅' if verdict['decision'] == 'APPROVED' else '❌'

print('Quality Sign-off Report')
print('------------------------')
print(f"Total students tested   : {verdict['total_students_tested']}")
print(f"Recommendation accuracy  : {verdict['top1_accuracy']*100:.1f}%")
print(f"Precision@5              : {current_metrics['precision_at_5']}")
print(f"Recall@5                  : {current_metrics['recall_at_5']}")
print(f"False positive rate@5     : {current_metrics['fpr_at_5']}")
print(f"Precision drop vs baseline: {verdict['precision_drop']:+.3f}")
print(f"Regression detected       : {'Yes' if verdict['regression_detected'] else 'No'}")
print()
print(f"{symbol} Decision: {verdict['decision']}")


**Matching quality signed off.** The model is unchanged from Task 7, the metrics hold up
on a fresh, much larger sample of platform traffic, and Section 4 above is the evidence
that this sign-off process would have actually caught it if something had broken.
